In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image


In [ ]:
# Install Kaggle API
!pip install -q kaggle

# Set up Kaggle credentials (users will enter their own keys here)
import os
os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"  # Replace with your Kaggle username
os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_KEY"            # Replace with your Kaggle API key

# Download and unzip the dataset directly into Colab
!kaggle datasets download -d yusufmurtaza01/chest-xray-pneumonia-balanced-dataset
!unzip -q chest-xray-pneumonia-balanced-dataset.zip -d /content/dataset

dataset_path = "/content/dataset"
print(os.listdir(dataset_path))

Found dataset folders: ['test', 'train', 'val']


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
train_transform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomCrop(224),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

NameError: name 'transforms' is not defined

In [5]:
train_data = datasets.ImageFolder(dataset_path + "/train", transform=train_transform)
val_data   = datasets.ImageFolder(dataset_path + "/val", transform=test_transform)
test_data  = datasets.ImageFolder(dataset_path + "/test", transform=test_transform)

class_names = train_data.classes

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))
print(class_names)

Train: 6800
Val: 1700
Test: 30
['NORMAL', 'PNEUMONIA']


In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32)
test_loader  = DataLoader(test_data, batch_size=32)

# For TRAIN confusion matrix
train_eval_data = datasets.ImageFolder(dataset_path + "/train", transform=test_transform)
train_eval_loader = DataLoader(train_eval_data, batch_size=32, shuffle=False)

In [7]:
def get_predictions(model, loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            preds.extend(predicted.cpu().numpy())
            labels.extend(targets.numpy())

    return labels, preds

In [ ]:
def train_and_evaluate_model(model,
                             model_name,
                             train_loader,
                             train_eval_loader,
                             val_loader,
                             test_loader,
                             num_epochs=20,
                             lr=0.001):

    print("\n" + "="*50)
    print("MODEL:", model_name)
    print("="*50)

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    # ================= TRAINING =================
    for epoch in range(num_epochs):

        model.train()

        total, correct, loss_sum = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            loss_sum += loss.item()

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_acc = correct / total
        train_loss = loss_sum / len(train_loader)

        # validation
        model.eval()

        v_total, v_correct, v_loss = 0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                v_loss += loss.item()

                _, preds = torch.max(outputs, 1)

                v_total += labels.size(0)
                v_correct += (preds == labels).sum().item()

        val_acc = v_correct / v_total
        val_loss = v_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Val Acc: {val_acc:.4f}")

    # test
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # performance metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted')
    rec = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    print("\nTEST RESULTS")
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1:", f1)

    print("\nClassification Report")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # Train confusion matrix
    t_labels, t_preds = get_predictions(model, train_eval_loader)

    cm_train = confusion_matrix(t_labels, t_preds)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm_train, annot=True, fmt="d", cmap="Greens",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(model_name + " - TRAIN CM")
    plt.show()

    # Test confusion matrix
    cm_test = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(model_name + " - TEST CM")
    plt.show()

    return model

In [ ]:
densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

for param in densenet.parameters():
    param.requires_grad = False

densenet.classifier = nn.Linear(1024, 2)

for param in densenet.classifier.parameters():
    param.requires_grad = True

trained_densenet = train_and_evaluate_model(
    model=densenet,
    model_name="DenseNet-121",
    train_loader=train_loader,
    train_eval_loader=train_eval_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    num_epochs=20
)
torch.save(trained_densenet.state_dict(), "densenet121.pth")

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\KTS/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth


13.8%

In [ ]:
efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

for param in efficientnet.parameters():
    param.requires_grad = False

efficientnet.classifier[1] = nn.Linear(1280, 2)

for param in efficientnet.classifier[1].parameters():
    param.requires_grad = True

trained_efficientnet = train_and_evaluate_model(
    model=efficientnet,
    model_name="EfficientNet-B0",
    train_loader=train_loader,
    train_eval_loader=train_eval_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    num_epochs=20
)
torch.save(trained_efficientnet.state_dict(),
           "efficientnet_b0.pth")

In [ ]:
sample_image_path = "sample.jpeg" 

if os.path.exists(sample_image_path):
    img = Image.open(sample_image_path).convert("RGB")
    img_tensor = test_transform(img).unsqueeze(0).to(device)

    # DenseNet prediction
    trained_densenet.eval()
    with torch.no_grad():
        output = trained_densenet(img_tensor)
        _, pred = torch.max(output, 1)
    print(f"DenseNet Prediction: {class_names[pred.item()]}")

    # EfficientNet prediction
    trained_efficientnet.eval()
    with torch.no_grad():
        output = trained_efficientnet(img_tensor)
        pred = torch.max(output, 1)
    print(f"EfficientNet Prediction: {class_names[pred.item()]}")
else:
    print(f"Could not find '{sample_image_path}' to run inference. Add an image to test single predictions.")